# sum-and-broadcast-duality composite — cx14: row-mean as sum-along-axis then broadcast-divide

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `broadcasting-rules`, `sum-and-broadcast-duality`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "sum-and-broadcast-duality"
DD_ATOM_IDS = ["broadcasting-rules", "sum-and-broadcast-duality"]
DD_SUBTOPICS = ["Numpy: Vectorization and broadcasting", "Backprop: sum/broadcast duality"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Sum + broadcast = mean — two atoms in one expression

1. **`broadcasting-rules`** — the right-align rule that lets a `(B, 1)`
   tensor multiply or divide across `(B, D)` without an explicit loop.
2. **`sum-and-broadcast-duality`** — `sum(dim=-1, keepdim=True)` reduces an
   axis to size 1; the kept axis is exactly the shape you need to broadcast
   the result back. `mean = sum / count` is the canonical composition.

Composition: every "reduce then re-distribute" pattern (mean, normalize,
softmax, layernorm) is shaped like this. The `keepdim=True` is what makes
the broadcast step a one-liner.


### Composite Exercise — row-mean as sum-along-axis then broadcast-divide

**Atoms exercised together**: `broadcasting-rules`, `sum-and-broadcast-duality`

Implement `cx14_row_mean_normalize(x)`. Given `x` shape `(B, D)`:

1. Compute the per-row sum with `keepdim=True` → shape `(B, 1)`.
2. Divide that sum by `D` (the row length) to get the per-row mean. Both
   `sum / D` and `sum / x.shape[-1]` are fine.
3. Return a tensor of shape `(B, D)` where every entry in row `i` equals the
   mean of row `i`. (`mean.expand_as(x)` or just relying on broadcast via
   `x * 0 + mean` both work — the test only checks the values.)

Constraints:
- Use `sum` along `dim=-1` with `keepdim=True` (not `.mean(...)` directly).
  The whole point of this drill is wiring sum + broadcast by hand.
- Output must equal `x.mean(dim=-1, keepdim=True).expand_as(x)` value-wise.


In [ ]:
def cx14_row_mean_normalize(x: Tensor) -> Tensor:
    # atom: sum along last axis with keepdim → (B, 1)
    row_sum = x.sum(dim=-1, keepdim=True)
    # mean = sum / count
    row_mean = row_sum / x.shape[-1]
    # atom: broadcasting-rules — (B, 1) right-aligns against D and expands
    return row_mean.expand_as(x)


<details><summary>Show solution — cx14</summary>

```python
def cx14_row_mean_normalize(x: Tensor) -> Tensor:
    # atom: sum along last axis with keepdim → (B, 1)
    row_sum = x.sum(dim=-1, keepdim=True)
    # mean = sum / count
    row_mean = row_sum / x.shape[-1]
    # atom: broadcasting-rules — (B, 1) right-aligns against D and expands
    return row_mean.expand_as(x)

```

`x.sum(dim=-1, keepdim=True)` is the sum-half of the duality; the divide-by-D
is the count step; `.expand_as(x)` (or any broadcast op against `x`) is the
broadcasting-rules step. Drop the `keepdim` and the divide would need an
extra `unsqueeze` to broadcast back.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx14'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx14',
        'subtopics': ["Numpy: Vectorization and broadcasting", "Backprop: sum/broadcast duality"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()